# Apache Spark — Core Concepts & Optimization

Training notebook covering Spark internals: distributed processing, lazy evaluation, execution model, performance monitoring (Serverless tools and the classic-compute Spark UI & logs), transformations, streaming, and optimization techniques.

## 📋 What We'll Cover

1. **Distributed Processing Architecture** — How Spark distributes work across Driver, Executors, and Tasks
2. **Lazy Evaluation** — Transformations vs Actions, and reading Execution Plans (Parsed → Analyzed → Optimized → Physical)
3. **Narrow vs Wide Transformations** — Independent partition work vs shuffles, and their impact on stages
4. **Performance Monitoring (Serverless)** — Using `explain()`, Query Profile, and Query History
5. **Spark UI & Logs (Classic Compute)** — Jobs, Stages, Storage, Executors tabs; spotting skew and spill
6. **Batch vs Streaming Workloads**
   * 6a. Batch Processing — Full reloads from cloud storage
   * 6b. Streaming with Auto Loader — Incremental file ingestion with `availableNow` trigger
   * 6c. Load Options — Triggers, bad-record handling, and schema evolution
7. **Join Types** — Inner, Left, Right, Full, Cross, Anti, Semi (Batch & Streaming)
8. **PySpark Do's and Don'ts** — Best practices and common pitfalls
9. **Cleanup** — Drop tables and clear checkpoints

In [0]:
%sql
USE CATALOG training;
USE SCHEMA bronze;
 SELECT current_catalog() AS catalog, current_schema() AS schema;

## 1. Distributed Processing Architecture

### How Spark Distributes Work

```
┌─────────────────────────────────────────────────────────┐
│                     DRIVER PROGRAM                      |
│  • Schedules tasks                • Collects results    │
└──────────────────────┬──────────────────────────────────┘
                       │  distributes tasks
       ┌───────────────┼───────────────┐
       ▼               ▼               ▼
 ┌───────────┐   ┌───────────┐   ┌───────────┐
 │ EXECUTOR 1│   │ EXECUTOR 2│   │ EXECUTOR 3│
 │ ┌───┬───┐ │   │ ┌───┬───┐ │   │ ┌───┬───┐ │
 │ │T1 │T2 │ │   │ │T3 │T4 │ │   │ │T5 │T6 │ │
 │ └───┴───┘ │   │ └───┴───┘ │   │ └───┴───┘ │
 └───────────┘   └───────────┘   └───────────┘
    Worker 1        Worker 2        Worker 3
```

### Key Concepts

| Component | Role |
|---|---|
| **Driver** | Orchestrates the application, creates DAG, assigns tasks |
| **Executor** | JVM process on worker node, runs tasks & stores data |
| **Task** | Smallest unit of work — operates on **one partition** |
| **Partition** | Chunk of data processed in parallel |
| **Cluster Manager** | Allocates resources (managed by Databricks on Serverless) |

> **Key insight:** More partitions = more parallelism (up to available cores)

In [0]:
from pyspark.sql.functions import col, sum as _sum

# --- Lazy: Nothing executes yet ---
orders = spark.read.table("samples.tpch.orders")



In [0]:

# Chain of transformations — all lazy
result = (
    orders
    .filter(col("o_orderstatus") == "F")
    .groupBy("o_orderpriority")
    .agg(_sum("o_totalprice").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)

print("✅ No computation happened yet — only a plan was built")
print("─" * 55)

# View the execution plan
result.explain(mode="formatted")

In [0]:
# --- Trigger execution with an ACTION ---
# This creates: 1 Job → multiple Stages → Tasks
# Check the Spark UI after running!

display(result)  # ACTION: triggers the full DAG

In [0]:
#Why it does laze evaluation , Because of the lazy evaluation, Spark can optimize the query plan before executing it. For example, it can push down filters to the data source, or it can optimize joins to reduce the amount of data that needs to be transferred between nodes.
df=spark.read.table("samples.tpch.orders")


In [0]:
df.filter(col("o_orderstatus") == "F")


In [0]:
df.display()

### Cache vs Persist — Why do we need them?

Because of **lazy evaluation**, a DataFrame is only a *recipe* (plan). Every **action** (`count`, `display`, `write`…) re-runs the **whole recipe from the source** — read, join, aggregate — again and again.

If you use the **same expensive DataFrame in several actions**, Spark repeats that work each time. `cache()` / `persist()` tell Spark: *"after you compute this once, keep the result and reuse it"*.

| | `cache()` | `persist(StorageLevel)` |
|---|---|---|
| What it is | Shortcut for `persist()` with the default level | Same thing, but **you choose where** data is kept |
| Default storage | `MEMORY_AND_DISK` for DataFrames | Any level: `MEMORY_ONLY`, `MEMORY_AND_DISK`, `DISK_ONLY`, `*_2` (replicated) … |
| Is it lazy? | Yes — data is stored on the **first action** | Yes — same |
| Release | `df.unpersist()` | `df.unpersist()` |

**When to use them**
* The same DataFrame is used **2+ times** (several actions, several joins, a loop, ML training iterations).
* The DataFrame is **expensive to build** (big joins, wide aggregations, complex UDFs) but the **result is small enough** to fit in cluster memory/disk.
* Interactive exploration — you will query the same filtered data many times.

## Persist vs Cache ():

Cache you can not choose where the dataframe needs to be stored, it will be chosen by spark as per the its version
Persist you can explicetly say whether the data needs to be stored in memory or in the disk, usually if the size is small , the memory will be appropriate

**When NOT to use them**
* The DataFrame is used **only once** — caching just adds extra work.
* Simple read + filter on Delta — Databricks **disk cache** and Delta data skipping already make re-reads fast.
* Data is huge compared to cluster memory — it causes spills/evictions and can be slower than recomputing.
* **Serverless compute** does not support `cache()`/`persist()` — write an intermediate Delta table instead.
* Always `unpersist()` when done, so memory is free for other work.

Check cached data in the **Spark UI → Storage** tab.

In [0]:
import time
from pyspark.sql.functions import col, sum as _sum, count as _count
from pyspark import StorageLevel

# An "expensive" DataFrame: join + aggregation (only a recipe, nothing runs yet)
orders   = spark.read.table("samples.tpch.orders")
customer = spark.read.table("samples.tpch.customer")

cust_revenue = (orders.join(customer, orders.o_custkey == customer.c_custkey)
                      .groupBy("c_mktsegment", "c_nationkey")
                      .agg(_sum("o_totalprice").alias("revenue"),
                           _count("*").alias("order_cnt")))

def run_actions(df, label):
    spark.conf.set("spark.job.description", label)        # label shows in Spark UI Jobs tab
    start = time.time()
    df.count()                                            # action 1
    df.filter(col("revenue") > 1e9).count()               # action 2
    df.orderBy(col("revenue").desc()).limit(5).collect()  # action 3
    print(f"{label:<28}: {time.time() - start:6.2f} sec")

# 1) WITHOUT cache -> every action re-reads both tables, re-joins and re-aggregates
run_actions(cust_revenue, "Without cache")

# 2) WITH cache() -> computed once on first action, later actions read the stored result
cust_revenue.cache()
run_actions(cust_revenue, "With cache (1st = fill)")
run_actions(cust_revenue, "With cache (reuse)")
cust_revenue.unpersist()                                  # free memory

# 3) WITH persist() -> same idea, but you choose the storage level
cust_revenue.persist(StorageLevel.DISK_ONLY)
run_actions(cust_revenue, "With persist DISK_ONLY")
run_actions(cust_revenue, "With persist (reuse)")
print("Storage level:", cust_revenue.storageLevel)
cust_revenue.unpersist()

spark.conf.unset("spark.job.description")


### Transformations vs Actions

| Transformations (Lazy) | Actions (Trigger Execution) |
|---|---|
| `select`, `filter`, `withColumn` | `count`, `show`, `collect` |
| `groupBy`, `join`, `orderBy` | `write`, `save`, `display` |
| `map`, `flatMap`, `union` | `first`, `take`, `foreach` |
| Build a **logical plan** — nothing runs yet | **Trigger** the DAG → jobs → stages → tasks |

### Execution Hierarchy

```
Action (e.g. df.count())
 └── Job  (one per action)
      ├── Stage 1  (separated by shuffles)
      │    ├── Task 1  (one per partition)
      │    └── Task 2
      └── Stage 2
           ├── Task 3
           └── Task 4
```

### Execution Plans

| Plan | What It Shows |
|---|---|
| **Parsed Logical Plan** | Raw user intent |
| **Analyzed Logical Plan** | Resolved columns & types |
| **Optimized Logical Plan** | Catalyst optimizer rewrites |
| **Physical Plan** | Actual execution strategy (scans, joins, exchanges) |

> **Key insight:** Spark optimizes the **entire chain** of transformations before running anything

### Execution Plan Phases — What Happens at Each Plan

The Catalyst optimizer turns your DataFrame code into 4 plans. `explain(mode="extended")` prints all of them.

```
Your code → ① Parsed → ② Analyzed → ③ Optimized → ④ Physical → Jobs / Stages / Tasks
            (syntax)  (catalog)   (rules)       (how to run)
```

| Plan | Activity | What to look for in the example below |
|---|---|---|
| **① Parsed (Unresolved) Logical** | Builds a tree from your code exactly as written. Tables & columns are **not checked yet**. | `'UnresolvedRelation [samples, tpch, orders]`, columns prefixed with `'` (`'o_totalprice`), two separate `'Filter` nodes, `100 * 10` not computed |
| **② Analyzed Logical** | Resolves names against **Unity Catalog**: checks tables/columns exist, assigns data types and unique IDs, adds casts. Errors like `UNRESOLVED_COLUMN` come from here. | `Relation samples.tpch.orders[...] parquet`, `o_totalprice#2104`, output types `revenue: decimal(28,2)`, `cast(... as decimal(18,2))` |
| **③ Optimized Logical** | Applies rule-based rewrites — same result, less work. | **Constant folding**: `100 * 10` → `1000.00` • **Combine filters**: two filters → one `AND` • **Predicate pushdown**: filter moved *below* the join • **Column pruning**: `Project [o_custkey, o_totalprice]` before the join • **Null filtering**: `isnotnull(o_custkey)` added for the join key |
| **④ Physical** | Chooses **how** to execute: join algorithm, shuffles, scan details, Photon operators. AQE can still change it at runtime (`isFinalPlan=false`). | `PhotonScan ... ReadSchema` (only 3 columns read), `DataFilters` / `DictionaryFilters` (filter pushed into the file scan), `ShuffleExchange hashpartitioning(o_custkey, 200)` (= shuffle = new stage), `ShuffledHashJoin` vs `BroadcastHashJoin`, `partial_sum` → shuffle → `finalmerge_sum` |

> Each **Exchange** in the physical plan is a stage boundary you will see in the Spark UI after an action runs.

## 3. Narrow vs Wide Transformations

### Comparison

| | Narrow | Wide |
|---|---|---|
| **Data movement** | None — each partition works independently | **Shuffle** — data moves across nodes |
| **Stage boundary** | Same stage | Creates a **new stage** |
| **Examples** | `select`, `filter`, `map`, `withColumn`, `union` | `groupBy`, `join`, `orderBy`, `distinct`, `repartition` |
| **Performance** | ✅ Fast, pipelined | ⚠️ Expensive — network + disk I/O |

### Visual

```
NARROW (no shuffle)              WIDE (shuffle = stage boundary)

 Partition 1 → Partition 1       Partition 1 ─┬─→ Partition A
 Partition 2 → Partition 2       Partition 2 ─┼─→ Partition B
 Partition 3 → Partition 3       Partition 3 ─┴─→ Partition C
                                  ▲ data redistributed
```

> **Key insight:** Every wide transformation = a shuffle = network I/O = **the #1 performance bottleneck in Spark**

In [0]:
from pyspark.sql.functions import col, upper

orders = spark.read.table("samples.tpch.orders")

# --- NARROW transformations (no shuffle, same stage) ---
narrow_result = (
    orders
    .select("o_orderkey", "o_orderstatus", "o_totalprice")
    .filter(col("o_totalprice") > 1000)
    .withColumn("status_upper", upper(col("o_orderstatus")))
)

print("=== NARROW Plan (no Exchange/shuffle) ===")
narrow_result.explain()

print("\n" + "═" * 55)

# --- WIDE transformations (shuffle required) ---
wide_result = (
    orders
    .groupBy("o_orderstatus")
    .count()
    .orderBy(col("count").desc())
)

print("\n=== WIDE Plan (Exchange = shuffle!) ===")
wide_result.explain()

## 4. Performance Monitoring on Serverless

> ⚠️ **Serverless does NOT have the traditional Spark UI.** Use these tools instead. On **classic compute**, the Spark UI and driver/executor logs are available — see **Section 4b**.

### How to Monitor on Serverless

| Tool | What It Shows | How to Access |
|---|---|---|
| **`explain()`** | Physical plan — scans, joins, shuffles, filters | `df.explain()` or `df.explain(mode="formatted")` |
| **`explain("cost")`** | Plan + estimated row counts & sizes | `df.explain(mode="cost")` |
| **Query Profile** | Visual execution plan in SQL Editor | Run query in SQL Editor → click **"Query Profile"** |
| **Query History** | Past queries, duration, rows scanned | Left sidebar → **Query History** |

### Reading an Execution Plan

```
AdaptiveSparkPlan          ← AQE wrapper (always present)
 +- PhotonSort              ← orderBy()
    +- Exchange              ← SHUFFLE (wide transformation!)
       +- PhotonGroupingAgg  ← groupBy().agg()
          +- PhotonScan      ← table/file scan (filters pushed here)
```

### What to Look For in `explain()`

| Look For | Meaning |
|---|---|
| **PhotonScan** with `DataFilters` | ✅ Predicate pushdown is working |
| **Exchange** | ⚠️ Shuffle happening — wide transformation |
| **PhotonBroadcastHashJoin** | ✅ Small table broadcast (no shuffle) |
| **SortMergeJoin** | ⚠️ Both sides shuffled — expensive |
| **AdaptiveSparkPlan** | ✅ AQE is active |
| **ReadSchema** with fewer columns | ✅ Column pruning is working |

> 💡 **Tip:** For visual execution plans, run your query in the **SQL Editor** and click **"Query Profile"**

## 4b. Spark UI & Logs on Classic Compute

> Applies to **classic all-purpose / job clusters** (Dedicated or No-isolation access mode). Serverless and Standard (shared) compute do not expose the Spark UI — use Section 4 instead.

### How to Open the Spark UI

| From | Path |
|---|---|
| **Notebook cell** | Expand **"Spark Jobs"** under a cell's output → click **View** next to a job |
| **Cluster page** | **Compute** → select cluster → **Spark UI** tab |
| **Job run** | **Jobs & Pipelines** → run → task → **Spark UI** link |
| **Terminated cluster** | Spark UI stays available (event-log based) for a limited time after termination |

### Spark UI Tabs

| Tab | What It Shows | Use It To |
|---|---|---|
| **Jobs** | One row per action; duration, stages, status | Find the slow action |
| **Stages** | Tasks per stage, shuffle read/write, spill, task time distribution (min/median/max) | Detect **skew** (max >> median) and **spill** |
| **Storage** | Cached DataFrames/tables, memory vs disk usage | Verify caching worked / free memory |
| **Environment** | Spark configs, JVM & classpath | Confirm `spark.sql.*` settings |
| **Executors** | Cores, memory, GC time, failed tasks, **stdout/stderr per executor** | Spot GC pressure, lost executors, executor logs |
| **SQL / DataFrame** | Visual physical plan with **runtime metrics** per operator (rows, bytes, time) | See actual rows per Exchange/Join, AQE final plan |
| **Structured Streaming** | Input rate, process rate, batch duration per stream | Check whether a stream keeps up |

### Execution Hierarchy in the UI

```
Jobs tab      →  Job 12  (display(result))
Stages tab    →    ├── Stage 30  Scan + partial agg  (200 tasks)   shuffle write
              →    └── Stage 31  Final agg + sort    (AQE: 4 tasks) shuffle read
SQL tab       →  Query 7: visual DAG with rows/bytes per operator
```

### Red Flags to Look For

| Symptom (where) | Likely Cause | Fix |
|---|---|---|
| Max task time >> median (Stages → Summary Metrics) | **Data skew** | AQE skew join, salting, filter hot keys |
| **Spill (Memory/Disk)** > 0 (Stages) | Partitions too large for executor memory | More shuffle partitions, larger nodes |
| Large **Shuffle Read/Write** (Stages) | Wide transformations, big joins | Broadcast small tables, filter/prune earlier |
| High **GC Time** (Executors) | Memory pressure, large objects | Bigger memory, fewer cached objects |
| Thousands of tiny tasks (Stages) | Too many small files/partitions | `OPTIMIZE`, coalesce, AQE coalescing |
| Failed/killed executors (Executors) | OOM, spot instance loss | Check executor stderr / driver log4j |

> 💡 **Tip:** Label your jobs with `spark.conf.set("spark.job.description", "...")` (next cell) so they are easy to find in the Jobs and SQL tabs.

In [0]:
from pyspark.sql.functions import col, sum as _sum, count

# spark.job.description sets the 'Description' column in the Spark UI Jobs / SQL tabs.
# Works on Dedicated AND Standard (shared) access mode — no sparkContext needed.

orders = spark.read.table("samples.tpch.orders")

# Job 1: narrow + wide (filter -> groupBy -> orderBy)
spark.conf.set("spark.job.description", "dox_01_revenue_by_priority")
revenue = (
    orders.filter(col("o_orderstatus") == "F")
)
display(revenue)


In [0]:
from pyspark.sql.functions import col, when, count, sha2

# --- Create an intentionally skewed key ---
# ~90% of rows land on key 0 -> one shuffle partition gets most of the data.
skewed = (
    spark.range(5_000_000)
    .withColumn("key", when(col("id") % 10 < 9, 0).otherwise(col("id") % 1000))
    .withColumn("payload", sha2(col("id").cast("string"), 256))   # make rows heavier
)

# NOTE: groupBy("key").count() HIDES the skew in the Spark UI.
# Spark does a partial (map-side) aggregation BEFORE the shuffle, so each task sends
# only ~1 row per key -> shuffle partitions are tiny and evenly sized.

# repartition by key has NO pre-aggregation: ALL rows of key 0 go to ONE task.
spark.conf.set("spark.job.description", "dox_03_skewed_repartition")
skewed.repartition(200, "key").write.format("noop").mode("overwrite").save()   # noop = run the job, write nothing
spark.conf.unset("spark.job.description")

# Same thing in numbers: rows per shuffle partition
from pyspark.sql.functions import spark_partition_id
display(skewed.repartition(200, "key")
        .groupBy(spark_partition_id().alias("partition")).count()
        .orderBy(col("count").desc()).limit(10))

print("""💡 In the Spark UI (classic):
   Stages tab -> open the shuffle-read stage -> 'Summary Metrics for Tasks'
   • Job 'dox_03_skewed_repartition' -> 2nd stage (200 tasks)
   • Duration / Shuffle Read Size: Max is far above the Median => SKEW
   • Check 'Spill (Memory)' / 'Spill (Disk)' columns for spilling tasks
   SQL tab -> AQE may show 'AQEShuffleRead' with coalesced/skew-split partitions
""")

## 5. Batch Processing:

* Batch Processing is a scheduled compute paradigm used to process bulk datasets at predefined intervals (e.g., hourly, daily) rather than processing records continuously as they arrive.
* Simplifies complex multi-table joins, aggregations, and business transformations without the state-management overhead of real-time continuous streaming.
* Manual workaround needed to identify and process incremental


In [0]:
# ========== SHARED CONFIG for Section 5 (Batch + Streaming) ==========
# Source: 20 CSV part files (TPC-H customer, ~750K rows, with header)
SOURCE_PATH = "abfss://inboundfiles@unityms.dfs.core.windows.net/customer"

CATALOG, SCHEMA = "training", "bronze"

# Batch target
BATCH_TABLE = f"{CATALOG}.{SCHEMA}.customer_batch"

# Streaming targets — one table per trigger mode
STREAM_TABLE_AVAILABLE_NOW   = f"{CATALOG}.{SCHEMA}.customer_stream_available_now"
STREAM_TABLE_PROCESSING_TIME = f"{CATALOG}.{SCHEMA}.customer_stream_processing_time"
STREAM_TABLE_DEFAULT         = f"{CATALOG}.{SCHEMA}.customer_stream_default"

# Streaming state (checkpoints + Auto Loader schema) lives in a UC Volume
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.checkpoints")
CHECKPOINT_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/customer"

# Explicit schema — required for the plain file streaming source
CUSTOMER_SCHEMA = """
    c_custkey INT, c_name STRING, c_address STRING, c_nationkey INT,
    c_phone STRING, c_acctbal DOUBLE, c_mktsegment STRING, c_comment STRING
"""

print(f"Source     : {SOURCE_PATH}")
print(f"Batch table: {BATCH_TABLE}")
print(f"Checkpoints: {CHECKPOINT_ROOT}")


* `spark.read` reads **every file** in the folder each time it runs — it keeps **no memory** of what was already loaded.
* `mode("overwrite")` → full reload every run (safe but re-reads everything).
* `mode("append")` → re-running loads **the same files again** → duplicate rows.
* To make batch incremental you must build the tracking yourself (watermark columns, file lists, `MERGE`) — or use streaming (5b).

In [0]:
from pyspark.sql.functions import col, current_timestamp

# ========== BATCH READ ==========
# Bounded read: lists ALL files in SOURCE_PATH at execution time
batch_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(CUSTOMER_SCHEMA)
    .load(SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

# ========== BATCH WRITE ==========
(batch_df.write
    .format("delta")
    .mode("overwrite")          # full load
    .saveAsTable(BATCH_TABLE))

print(f"✅ Batch load #1 complete → {BATCH_TABLE}")
print(f"   Rows in table: {spark.table(BATCH_TABLE).count():,}")
display(spark.table(BATCH_TABLE).limit(5))

In [0]:
# ========== BATCH RE-RUN ==========
# Run the SAME batch read again. No new files arrived, but Spark has no state,
# so it re-reads all 20 files. With append, every row is loaded a second time.
rows_before = spark.table(BATCH_TABLE).count()

reprocess_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .schema(CUSTOMER_SCHEMA)
    .load(SOURCE_PATH)
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

(reprocess_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(BATCH_TABLE))

rows_after = spark.table(BATCH_TABLE).count()
print(f"Rows before re-run : {rows_before:,}")
print(f"Rows read again    : {rows_after - rows_before:,}   ← entire dataset re-loaded")
print(f"Rows after re-run  : {rows_after:,}")

# Each load batch is visible by _ingested_at; each customer now appears twice
display(
    spark.table(BATCH_TABLE)
    .groupBy("_ingested_at").count()
    .orderBy("_ingested_at")
)

# Delta history shows one WRITE per batch run
display(spark.sql(f"DESCRIBE HISTORY {BATCH_TABLE}").select("version", "timestamp", "operation", "operationParameters", "operationMetrics"))

### 5b. Streaming Processing (Incremental)

Same source folder, but `spark.readStream` + a **checkpoint** remembers which files were already processed. Re-running only picks up **new** files.

**Example below:** Auto Loader (`cloudFiles`) + `trigger(availableNow=True)` → `customer_stream_available_now`, then a re-run with the same checkpoint.

> Other trigger types and the read/write options (schema evolution, bad records) are covered in **Section 5c**.

In [0]:
# Read: Auto Loader (cloudFiles) — infers the schema and tracks processed files
df_available_now = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_ROOT}/available_now/_schema")
    .option("cloudFiles.maxFilesPerTrigger", 5)
    .load(SOURCE_PATH)
)

# Write: availableNow → process all files that exist now, then stop
query = (
    df_available_now.writeStream
    .option("checkpointLocation", f"{CHECKPOINT_ROOT}/available_now")
    .trigger(availableNow=True)
    .toTable(STREAM_TABLE_AVAILABLE_NOW)
)
query.awaitTermination()

print("Rows:", spark.table(STREAM_TABLE_AVAILABLE_NOW).count())

In [0]:
# Re-run Stream 1 with the SAME checkpoint → already-processed files are skipped
rows_before = spark.table(STREAM_TABLE_AVAILABLE_NOW).count()

df_rerun = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINT_ROOT}/available_now/_schema")
    .load(SOURCE_PATH)
)

query = (
    df_rerun.writeStream
    .option("checkpointLocation", f"{CHECKPOINT_ROOT}/available_now")
    .trigger(availableNow=True)
    .toTable(STREAM_TABLE_AVAILABLE_NOW)
)
query.awaitTermination()

rows_after = spark.table(STREAM_TABLE_AVAILABLE_NOW).count()
print("Rows before:", rows_before)
print("Rows after :", rows_after)
print("New rows   :", rows_after - rows_before, "(0 = nothing reloaded)")

### 5c. Load Options — Triggers, Bad Records & Schema Changes (and When to Use Them)

#### ① Streaming triggers — `writeStream.trigger(...)`

| Trigger | Behaviour | When to use |
|---|---|---|
| *(default — no trigger)* | Next micro-batch starts **as soon as** the previous one finishes. Runs until stopped | Low latency on an always-on cluster |
| `processingTime="10 seconds"` | One micro-batch **every interval**. Runs until stopped | Near-real-time with a predictable cost/latency trade-off |
| `availableNow=True` | Processes **all data available now** (in several batches, respecting `maxFilesPerTrigger`), then **stops** | Incremental batch on a schedule (Jobs) — the most common pattern |
| `once=True` *(deprecated)* | One single batch with everything, then stops. Ignores rate limits | Legacy code only — replace with `availableNow` |
| `continuous="1 second"` | Continuous processing, ms latency. Limited sources/sinks (e.g. Kafka), **no file sources / Delta sinks** | Rare. Prefer Real-Time Mode where available |
| `realTime=...` *(Real-Time Mode, newer DBR)* | Very low-latency streaming for Kafka/Kinesis-type sources | Operational, sub-second use cases |

**Related streaming options**

| Option | Purpose |
|---|---|
| `checkpointLocation` | Stores progress (processed files/offsets). **One per stream**. Delete it = reprocess everything |
| `maxFilesPerTrigger` / `cloudFiles.maxFilesPerTrigger` | Limit files per micro-batch |
| `maxBytesPerTrigger` / `cloudFiles.maxBytesPerTrigger` | Limit data size per micro-batch |
| `outputMode` (`append` / `update` / `complete`) | How results are written. `append` for ingestion, `complete`/`update` for aggregations |
| `queryName` | Name shown in `spark.streams.active` and the Streaming tab |
| File source `format("csv")` | Needs explicit `schema`. Re-lists the directory each batch |
| Auto Loader `format("cloudFiles")` | `cloudFiles.format`, `cloudFiles.schemaLocation`, `cloudFiles.inferColumnTypes`, `cloudFiles.schemaHints`. Scales to millions of files |

#### ② Read options (parsing CSV / JSON files — batch and streaming)

| Option | What happens to bad data | When to use |
|---|---|---|
| `mode = PERMISSIVE` *(default)* | Bad value → `null`, row kept. **Extra CSV columns are silently dropped** | Exploration, default |
| `mode = DROPMALFORMED` | Bad rows **silently dropped** | Only when losing rows is acceptable (rare) |
| `mode = FAILFAST` | Job **fails** on the first bad row | Strict data contracts, tests, financial data |
| `rescuedDataColumn` *(Databricks)* | Type mismatches, **extra columns**, and case mismatches go into a JSON column `_rescued_data`, and the row is kept | **Bronze ingestion — never lose data** |
| `badRecordsPath` *(Databricks)* | Corrupt records are written to a quarantine path as JSON | Quarantine for later reprocessing |
| `mergeSchema` *(Parquet/ORC/Avro read)* | Combines the different schemas across files | Reading Parquet folders written with evolving schemas |

#### ③ Write options (Delta table)

| Option | Behaviour | When to use |
|---|---|---|
| *(default)* **schema enforcement** | Write **fails** if columns/types don't match the table | Protect Silver/Gold tables |
| `.option("mergeSchema", "true")` | **Adds new columns**. Cannot change existing types | Additive changes from a trusted source |
| `.option("overwriteSchema", "true")` + `mode("overwrite")` | **Replaces** the schema (change types, drop or rename columns) | Intentional redesign + full reload |


#### Summary — Which Option to Choose

| Situation | Recommended option |
|---|---|
| Scheduled incremental loads | Auto Loader + `trigger(availableNow=True)` in a Job |
| Continuous low-latency ingestion | Default trigger or `processingTime` on a running cluster |
| Bronze ingestion, schema may change, must not lose data | Auto Loader + `addNewColumns` (default) + `_rescued_data`, run in a **Job with retries** |
| Target schema must stay fixed, still keep unexpected data | Auto Loader `rescue`, or `rescuedDataColumn` on batch reads |
| Strict contract, changes need human approval | `failOnNewColumns` (stream) / `FAILFAST` (batch) |
| Silver/Gold tables | Keep Delta **schema enforcement**. Use `mergeSchema` only for deliberate additive changes |
| Column type must change | `overwriteSchema` + full reload (or `delta.enableTypeWidening` for numeric widening) |
| Upserts with new source columns | `MERGE WITH SCHEMA EVOLUTION` |
| Bad rows must be kept aside for reprocessing | `badRecordsPath` |
| Avoid in production | `DROPMALFORMED` and PERMISSIVE without a rescue column. Both **lose data silently** |

## 6. Join Types — Batch & Streaming

### 6a. Batch join types

Example: `customer` (750K) joined to `orders` (7.5M) on `c_custkey = o_custkey`. In TPC-H about **1/3 of customers have no orders**, so the join types return very different results.

| Join type | `how=` | Returns | Typical use |
|---|---|---|---|
| **Inner** | `"inner"` *(default)* | Only matching rows from both sides | Customers **with** their orders |
| **Left outer** | `"left"` | All left rows + matches (nulls if no match) | Keep all customers, orders if any |
| **Right outer** | `"right"` | All right rows + matches | Keep all orders, customer if any |
| **Full outer** | `"full"` | All rows from both sides | Reconciliation between two systems |
| **Left semi** | `"left_semi"` | Left rows that **have** a match (left columns only, no duplicates) | "Customers who ordered" — like `WHERE EXISTS` |
| **Left anti** | `"left_anti"` | Left rows with **no** match | "Customers who never ordered", finding missing records |
| **Cross** | `.crossJoin()` | Every left row × every right row | Small dimension combinations only — explodes in size |

**How Spark executes a join (join strategies)**

| Strategy | When Spark picks it | Shuffle? |
|---|---|---|
| **Broadcast hash join** | One side is small (< `spark.sql.autoBroadcastJoinThreshold`, or `F.broadcast()` hint) | No — small side copied to every executor |
| **Sort-merge join** | Both sides large (default for big equi-joins) | Yes — both sides shuffled + sorted |
| **Shuffle hash join** | One side medium-size, AQE may switch to it | Yes |
| **Broadcast nested loop** | Non-equi conditions (`<`, `BETWEEN`) or cross joins | Can be very slow |

> 💡 AQE can switch sort-merge → broadcast at runtime and split skewed join keys automatically.

### Join Types — Explained with Small Data

Two small tables joined on **`customer.cust_id = orders.cust_id`**.

<table><tr><td valign="top">

**customer** (left)

| cust_id | name |
|---|---|
| 1 | Ahmet |
| 2 | Zeynep |
| 3 | Philippe |
| 4 | Cristina |

</td><td width="40"></td><td valign="top">

**orders** (right)

| cust_id | amount |
|---|---|
| 1 | 500 |
| 1 | 300 |
| 2 | 200 |
| 5 | 900 |

</td></tr></table>

Things to notice in the data:
* **Ahmet (1)** has **2 orders** → he appears twice when orders columns are returned.
* **Philippe (3)** and **Cristina (4)** have **no orders**.
* **cust_id 5** has an order but **no customer**.

---

#### Inner join → only matching rows (3 rows)

| c.cust_id | name | o.cust_id | amount |
|---|---|---|---|
| 1 | Ahmet | 1 | 500 |
| 1 | Ahmet | 1 | 300 |
| 2 | Zeynep | 2 | 200 |

#### Left outer join → all customers + matching orders (5 rows)

| c.cust_id | name | o.cust_id | amount |
|---|---|---|---|
| 1 | Ahmet | 1 | 500 |
| 1 | Ahmet | 1 | 300 |
| 2 | Zeynep | 2 | 200 |
| 3 | Philippe | *null* | *null* |
| 4 | Cristina | *null* | *null* |

#### Right outer join → all orders + matching customers (4 rows)

| c.cust_id | name | o.cust_id | amount |
|---|---|---|---|
| 1 | Ahmet | 1 | 500 |
| 1 | Ahmet | 1 | 300 |
| 2 | Zeynep | 2 | 200 |
| *null* | *null* | 5 | 900 |

#### Full outer join → everything from both sides (6 rows)

| c.cust_id | name | o.cust_id | amount |
|---|---|---|---|
| 1 | Ahmet | 1 | 500 |
| 1 | Ahmet | 1 | 300 |
| 2 | Zeynep | 2 | 200 |
| 3 | Philippe | *null* | *null* |
| 4 | Cristina | *null* | *null* |
| *null* | *null* | 5 | 900 |

#### Left semi join → customers who **have** an order (2 rows)

Only **left** columns, and each customer **once**, even though Ahmet has 2 orders (like `WHERE EXISTS`).

| cust_id | name |
|---|---|
| 1 | Ahmet |
| 2 | Zeynep |

#### Left anti join → customers with **no** order (2 rows)

Only **left** columns (like `WHERE NOT EXISTS`). Note: order 5 (no customer) is **not** shown — anti looks only at the left side.

| cust_id | name |
|---|---|
| 3 | Philippe |
| 4 | Cristina |

#### Cross join → every customer × every order (4 × 4 = 16 rows, no join condition)

| c.cust_id | name | o.cust_id | amount |
|---|---|---|---|
| 1 | Ahmet | 1 | 500 |
| 1 | Ahmet | 1 | 300 |
| 1 | Ahmet | 2 | 200 |
| 1 | Ahmet | 5 | 900 |
| 2 | Zeynep | 1 | 500 |
| … | … | … | … |
| 4 | Cristina | 5 | 900 |

---

#### Summary

| Join | Rows | Rule of thumb |
|---|---|---|
| Inner | 3 | Matches only |
| Left | 5 | All left, nulls where no match |
| Right | 4 | All right, nulls where no match |
| Full | 6 | All from both sides |
| Left semi | 2 | Left rows that match — no duplicates, left columns only |
| Left anti | 2 | Left rows that don't match |
| Cross | 16 | Left rows × right rows |

> **Duplicate keys multiply rows:** one customer with 2 orders gives 2 rows in inner/left/right/full joins. If the right side has many matches per key, the result can be much bigger than the left table.

In [0]:
from pyspark.sql import functions as F

customer = spark.table("samples.tpch.customer").select("c_custkey", "c_name", "c_nationkey")
orders   = spark.table("samples.tpch.orders").select("o_orderkey", "o_custkey", "o_totalprice")
nation   = spark.table("samples.tpch.nation").select("n_nationkey", "n_name")

cond = customer.c_custkey == orders.o_custkey

# Row count returned by each join type
join_counts = [
    (how, customer.join(orders, cond, how).count())
    for how in ["inner", "left", "right", "full", "left_semi", "left_anti"]
]
display(spark.createDataFrame(join_counts, "join_type STRING, rows_returned LONG"))

# Left anti: customers who never placed an order
display(customer.join(orders, cond, "left_anti").limit(5))

# Broadcast join: small dimension (25 rows) → no shuffle of customer
cust_nation = customer.join(F.broadcast(nation), customer.c_nationkey == nation.n_nationkey)
cust_nation.explain()   # look for BroadcastHashJoin in the plan

# Cross join: 3 customers × 2 nations = 6 rows
display(customer.limit(3).crossJoin(nation.limit(2)))

### 6b. Joins in Structured Streaming

#### Stream ↔ Static (e.g. streaming customers + `nation` dimension table)

* Each **micro-batch** of the stream is joined to the static table, like a normal batch join.
* **No state** is kept, so no watermark is needed.
* If the static side is a **Delta table**, every micro-batch reads its **latest version**. Updated dimension rows apply to **new** stream rows only — rows already written are **not** re-joined.
* Use `F.broadcast(static_df)` when the static table is small.

#### Stream ↔ Stream (e.g. `orders` events + `shipments` events)

* A row on one side may arrive before its match on the other side, so Spark **buffers both sides in state**.
* Without limits the state grows forever. Add **watermarks** on both sides + a **time-range condition** in the join so Spark knows when old rows can be dropped.
* Outer joins emit the unmatched (null) rows **only after the watermark passes** — results are delayed by the watermark delay.
* Stream-stream joins support **`append`** output mode only.

```python
orders_s = (spark.readStream.table("orders_events")
            .withWatermark("order_ts", "1 hour"))       # tolerate 1h late data
ships_s  = (spark.readStream.table("shipment_events")
            .withWatermark("ship_ts", "2 hours"))

joined = orders_s.join(
    ships_s,
    F.expr("""
        o_orderkey = s_orderkey AND
        ship_ts BETWEEN order_ts AND order_ts + INTERVAL 1 DAY   -- time bound for state cleanup
    """),
    "leftOuter")   # orders with no shipment within 1 day → emitted with nulls after the watermark
```

#### Which join types are supported in streaming

| Left ↔ Right | Inner | Left outer | Right outer | Full outer | Left semi / anti |
|---|---|---|---|---|---|
| **Static ↔ Static** (batch) | ✅ | ✅ | ✅ | ✅ | ✅ |
| **Stream ↔ Static** | ✅ stateless | ✅ stateless | ❌ | ❌ | ✅ semi (anti ✅ stateless) |
| **Static ↔ Stream** | ✅ stateless | ❌ | ✅ stateless | ❌ | ❌ |
| **Stream ↔ Stream** | ✅ watermark + time bound recommended | ⚠️ needs watermark on right + time bound | ⚠️ needs watermark on left + time bound | ⚠️ needs watermark + time bound | ⚠️ semi: watermark on right + time bound |

**Rule of thumb:** the **streaming side must be the "preserved" side** of an outer join with a static table, because Spark can never know that a static row will not be matched by future stream data.

#### When to use what

| Scenario | Join pattern |
|---|---|
| Enrich events with a lookup / dimension table | **Stream-static** (broadcast if small) |
| Correlate two event streams (orders ↔ payments, impressions ↔ clicks) | **Stream-stream** with watermarks + time bound |
| Dimension changes must update past results | Not a streaming join — use **batch** / `MERGE`, or SCD tables |
| Both sides huge, no time relation | **Batch** join on a schedule |

In [0]:
# Stream: new rows from the Auto Loader table built in 5b (Delta table as streaming source)
# Static: samples.tpch.nation (25 rows) → broadcast, re-read each micro-batch
STREAM_STATIC_TABLE = f"{CATALOG}.{SCHEMA}.customer_stream_with_nation"

customer_stream = spark.readStream.table(STREAM_TABLE_AVAILABLE_NOW)
nation_static   = spark.table("samples.tpch.nation").select("n_nationkey", "n_name")

enriched = customer_stream.join(
    F.broadcast(nation_static),
    customer_stream.c_nationkey == nation_static.n_nationkey,
    "left",          # stream on the LEFT → left outer is allowed
)

query = (
    enriched.writeStream
    .option("checkpointLocation", f"{CHECKPOINT_ROOT}/stream_static_join")
    .trigger(availableNow=True)
    .toTable(STREAM_STATIC_TABLE)
)
query.awaitTermination()

display(spark.table(STREAM_STATIC_TABLE).groupBy("n_name").count().orderBy("n_name"))

## 7. PySpark Do's and Don'ts

Simple rules that avoid the most common performance problems and failures. Examples use `samples.tpch.customer` (`df`) and `samples.tpch.orders`.

#### 1. Use built-in functions, not Python UDFs
```python
# ❌ Don't
from pyspark.sql.functions import udf
upper_udf = udf(lambda s: s.upper())
df.withColumn("name_upper", upper_udf("c_name"))

# ✅ Do
from pyspark.sql import functions as F
df.withColumn("name_upper", F.upper("c_name"))
```
**Why:** A Python UDF sends every row from the JVM to a Python process and back (serialization). Catalyst can't optimize inside it, and Photon can't run it. Built-ins (`F.when`, `F.regexp_replace`, `F.to_date`, array/map functions...) run natively and are often **10x+ faster**. Use a UDF only when no built-in exists (and then prefer a `pandas_udf`).

#### 2. Don't `collect()` / `toPandas()` large data
```python
# ❌ Don't
rows = spark.table("samples.tpch.orders").collect()      # 7.5M rows → driver memory
pdf  = spark.table("samples.tpch.orders").toPandas()

# ✅ Do
display(spark.table("samples.tpch.orders").limit(100))     # look at data
spark.table("samples.tpch.orders").write.saveAsTable("...") # save results
segments = [r.c_mktsegment for r in df.select("c_mktsegment").distinct().collect()]  # small result: OK
```
**Why:** `collect()` pulls all data from every executor into the **single driver** → out-of-memory errors and lost parallelism. Collect only small, already-aggregated results.

#### 3. Don't loop over rows in Python
```python
# ❌ Don't
total = 0
for r in df.collect():
    if r.c_mktsegment == "BUILDING":
        total += r.c_acctbal

# ✅ Do
df.filter(F.col("c_mktsegment") == "BUILDING").agg(F.sum("c_acctbal")).show()
```
**Why:** The loop runs on one machine, one row at a time. The DataFrame version runs in parallel across all executors.

#### 4. Don't call `withColumn` in a long loop
```python
# ❌ Don't
for c in df.columns:
    df = df.withColumn(c, F.trim(F.col(c).cast("string")))

# ✅ Do
df = df.select([F.trim(F.col(c).cast("string")).alias(c) for c in df.columns])
# or: df.withColumns({"a": ..., "b": ...})
```
**Why:** Each `withColumn` adds a new projection to the plan. Hundreds of them make the plan huge and analysis very slow (can even cause `StackOverflowError`). One `select` = one projection.

#### 5. Filter early, select only needed columns
```python
# ❌ Don't
orders = spark.table("samples.tpch.orders")
result = orders.join(df, orders.o_custkey == df.c_custkey).filter("o_orderstatus = 'F'")

# ✅ Do
orders_f = (spark.table("samples.tpch.orders")
            .filter("o_orderstatus = 'F'")
            .select("o_orderkey", "o_custkey", "o_totalprice"))
result = orders_f.join(df.select("c_custkey", "c_name"), orders_f.o_custkey == df.c_custkey)
```
**Why:** Less data is read, shuffled and joined. The optimizer often pushes filters down for you, but being explicit keeps it predictable — especially after UDFs or complex logic it can't see through.

#### 6. Give a schema instead of `inferSchema` on large files
```python
# ❌ Don't
spark.read.option("header", True).option("inferSchema", True).csv(SOURCE_PATH)

# ✅ Do
spark.read.option("header", True).schema(CUSTOMER_SCHEMA).csv(SOURCE_PATH)
```
**Why:** `inferSchema` reads the data an **extra time** just to guess types, and the guess can change between runs.

#### 7. Broadcast small lookup tables in joins
```python
# ✅ Do
nation = spark.table("samples.tpch.nation")   # 25 rows
df.join(F.broadcast(nation), df.c_nationkey == nation.n_nationkey)
```
**Why:** The small table is copied to every executor, so the big table is **not shuffled**. (AQE often does this automatically for small tables.)

#### 8. Check for empty data without `count()`
```python
# ❌ Don't
if df.count() > 0: ...

# ✅ Do
if not df.isEmpty(): ...
```
**Why:** `count()` scans everything. `isEmpty()` stops at the first row.

#### 9. Cache only what is reused — and release it
```python
# ❌ Don't
df = spark.table("samples.tpch.orders").cache()   # used once

# ✅ Do
hot = spark.table("samples.tpch.orders").filter("o_orderstatus = 'F'").cache()
hot.count()          # materialize once, then reuse several times
# ...
hot.unpersist()
```
**Why:** Caching costs memory and time. It only pays off when the same DataFrame is used multiple times. On Serverless, `cache()` is not supported — Delta caching is automatic.

#### 10. Control output file count
```python
# ❌ Don't
big_df.coalesce(1).write.parquet(path)          # 1 task writes everything
big_df.repartition(5000).write.parquet(path)    # thousands of tiny files

# ✅ Do
big_df.write.format("delta").saveAsTable("...")  # Delta + optimized writes / auto compaction
```
**Why:** One file = no parallelism. Too many small files = slow reads later. Let Delta size files, or pick a reasonable partition count.

#### 11. Import functions as a module
```python
# ❌ Don't
from pyspark.sql.functions import *    # shadows Python's sum, max, min, round, abs...

# ✅ Do
from pyspark.sql import functions as F
df.agg(F.max("c_acctbal"))
```
**Why:** `import *` silently replaces Python built-ins, causing confusing errors like `sum([1, 2])` failing.

#### 12. Use parameters, not f-strings, in `spark.sql`
```python
# ❌ Don't
spark.sql(f"SELECT * FROM samples.tpch.customer WHERE c_mktsegment = '{user_input}'")

# ✅ Do
spark.sql("SELECT * FROM samples.tpch.customer WHERE c_mktsegment = :seg",
          args={"seg": user_input})
```
**Why:** f-strings allow **SQL injection** and break on quotes in values. Named parameters are safe and type-correct.

#### Quick reference

| ❌ Avoid | ✅ Prefer | Main reason |
|---|---|---|
| Python `udf` | Built-in `F.*` functions (or `pandas_udf`) | Serialization, no Catalyst/Photon |
| `collect()` / `toPandas()` on big data | `display`, `limit`, `write` | Driver out-of-memory |
| Python loops over rows | DataFrame transformations | No parallelism |
| `withColumn` in loops | One `select` / `withColumns` | Plan bloat |
| Late filters, `select("*")` | Filter early, select needed columns | Less I/O and shuffle |
| `inferSchema` | Explicit schema | Extra scan, unstable types |
| Shuffle join with tiny table | `F.broadcast(small_df)` | Avoids shuffle |
| `count() > 0` | `isEmpty()` | Full scan |
| Caching everything | Cache only reused data, `unpersist()` | Memory pressure |
| `coalesce(1)` / huge `repartition` | Delta writes, sensible partitions | Parallelism / small files |
| `from ... import *` | `functions as F` | Name shadowing |
| f-string SQL | `spark.sql(..., args=...)` | SQL injection |

## 8. Cleanup

Drops every table created in this notebook and clears the streaming checkpoints. Without the checkpoint cleanup, a re-run would skip already-processed files and load **0 rows** into the new tables.

* **Not deleted:** the source CSV files in `abfss://inboundfiles@.../customer` and the `checkpoints` volume itself.
* Tables from older versions of this notebook (removed cells) are included with `IF EXISTS`, so the cell is safe to run at any time.

In [0]:
# Self-contained: works even if the setup cell was not run in this session
CATALOG, SCHEMA = "one_dbx_training", "default"
CHECKPOINT_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints/customer"

tables_to_drop = [
    "customer_batch",                   # 5a  batch load + append re-run
    "customer_stream_available_now",    # 5b  Auto Loader availableNow stream
    "customer_stream_with_nation",      # 6b  stream-static join
    # created by earlier versions of this notebook (cells since removed)
    "customer_stream_processing_time",
    "customer_stream_default",
    "customer_schema_demo",
    "customer_al_addnewcolumns",
    "customer_al_rescue",
    "customer_al_failonnewcolumns",
    "customer_al_none",
]

# Stop any streams still running before dropping their tables
for q in spark.streams.active:
    print(f"Stopping stream {q.name or q.id}")
    q.stop()

for t in tables_to_drop:
    full_name = f"{CATALOG}.{SCHEMA}.{t}"
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    print(f"Dropped (if existed): {full_name}")

# Clear checkpoints + Auto Loader schema location so streams start fresh next time
try:
    dbutils.fs.rm(CHECKPOINT_ROOT, True)
    print(f"Removed checkpoints: {CHECKPOINT_ROOT}")
except Exception as e:
    print(f"No checkpoints to remove: {str(e).splitlines()[0][:120]}")

# Optional: also drop the checkpoints volume
# spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.checkpoints")

# Verify: none of the demo tables should remain
remaining = [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}").collect()
             if r.tableName in tables_to_drop]
print("Remaining demo tables:", remaining or "none ✅")